### DATA INGESTION

In [ ]:
## Document Structure
from langchain_core.documents import Document

In [ ]:
doc=Document(
    page_content="this is the main text content I am using to create RAG pipeline",
    metadata={
        "source":"example.txt",
        "page":1,
        "author":"MD Ali",
        "date_created":"2026-04-02"
    }
)
doc

In [ ]:
## Create a simple text file
import os
os.makedirs("../data/text_file",exist_ok=True)

In [ ]:

sample_texts={
    "../data/text_file/python_intro.txt":"""Python Programming Introduction
    Python is a high-level, interpreted programming language that is easy to learn and use.
    It is widely used for web development, data analysis, artificial intelligence, and more.

    Key Features:
    - Easy to read and write
    - Strong community support
    - Versatile and powerful
    
    Python is widely used in various fields including:
    - Web development
    - Data analysis
    - Artificial Intelligence
    - Scientific computing
    """,
    "../data/text_file/machine_learning.txt":"""Machine Learning Basics
    Machine Learning is a field of artificial intelligence that focuses on building systems that learn from data.
    It involves training models to make predictions or decisions based on input data.

    Key Concepts:
    - Supervised Learning
    - Unsupervised Learning
    - Reinforcement Learning

    Machine Learning is used in various applications such as:
    - Image classification
    - Natural Language Processing
    - Recommendation Systems
    """
}

for filepath,content in sample_texts.items():
    with open(filepath,"w",encoding="utf-8") as f:
        f.write(content)
print("Text files created successfully")

In [ ]:
### TextLoader
from langchain_community.document_loaders import TextLoader

In [ ]:
loader=TextLoader("../data/text_file/python_intro.txt",encoding="utf-8")
documents=loader.load()
print(documents)

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader(
    "../data/text_file",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"},
    show_progress=False
    )
documents=dir_loader.load()
documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)
print(len(chunks))

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

In [ ]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
# Load documents
documents = dir_loader.load()

# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))


# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)


# Vector DB
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)


# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


# Query
query = "What are key features of Python?"

results = retriever.invoke(query)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:")
    print(r.page_content)